# 1.Setup

In [1]:
import pandas as pd
import polars as pl
import polars.selectors as cs

import numpy as np

import altair as alt

from collections import Counter

from util import funcoes_exploracao as fe  # funcoes que criei

from sc_detect.config import RAW_DATA_DIR



2026-10-20 11:34:01.950 | INFO     | sc_detect.config:<module>:11 - PROJ_ROOT path is: /home/gabriel_santiago/Estudos/Portfolio/skin_cancer_detection


In [2]:
# definindo a seed
pl.set_random_seed(42)

In [3]:
# configuracoes altair
alt.renderers.enable("default")
alt.data_transformers.enable("vegafusion", max_rows=None)
alt.theme.enable("fivethirtyeight")

ThemeRegistry.enable('fivethirtyeight')

# 0.Pequeno Teste

Vou testar Polars ao invés de Pandas pela primeira vez, então, aproveitarei para fazer algumas comparações simples e rápidas.

In [4]:
train_metadados_pd = pd.read_csv(RAW_DATA_DIR / 'train-metadata.csv')

/tmp/ipykernel_22143/100571722.py:1: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  train_metadados_pd = pd.read_csv(RAW_DATA_DIR / 'train-metadata.csv')


In [5]:
train_metadados = pl.read_csv(RAW_DATA_DIR / 'train-metadata.csv')

In [6]:
train_metadados_pd.head()

,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,0,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,0,IP_4111386,65.0,male,anterior torso,3.22,TBP tile: close-up,3D: XP,14.242329,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,0,IP_8313778,55.0,male,anterior torso,2.73,TBP tile: close-up,3D: white,24.725520,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510


In [7]:
train_metadados.head()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,i64,str,str,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""ISIC_0015670""",0,"""IP_1235828""","""60""","""male""","""lower extremity""",3.04,"""TBP tile: close-up""","""3D: white""",20.244422,16.261975,26.922447,23.954773,33.684638,28.953117,53.058545,55.828924,54.367448,62.025701,3.152561,27.47617,0.0,3.982447,2.967674,-7.658253,8.360566,5.784302,0.901302,"""Right Leg - Upper""","""Right Leg""",1.543016,0.002629,7.09136,0.0,9.307003,0.0,2.036195,2.63778,0.590476,85,-182.703552,613.493652,-42.427948,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",97.517282
"""ISIC_0015845""",0,"""IP_8170065""","""60""","""male""","""head/neck""",1.1,"""TBP tile: close-up""","""3D: white""",31.71257,25.36474,26.331,24.54929,41.21903,35.29926,39.70291,44.06404,48.86152,55.36236,0.9194971,12.23529,0.0,6.34783,1.781713,-6.500838,6.839008,4.987244,0.639885,"""Head & Neck""","""Head & Neck""",0.8219178,1.3343e-7,2.116402,0.0,3.354148,0.0,0.8532267,3.912844,0.2857143,55,-0.078308,1575.687,57.1745,"""Memorial Sloan Kettering Cance…","""CC-BY""","""IL_6727506""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",3.141455
"""ISIC_0015864""",0,"""IP_6724798""","""60""","""male""","""posterior torso""",3.4,"""TBP tile: close-up""","""3D: XP""",22.57583,17.12817,37.97046,33.48541,44.17492,37.6118,59.26585,62.90973,53.96118,61.67052,3.265153,24.18462,0.0,5.447655,4.485044,-7.709336,9.092376,6.290359,0.932147,"""Torso Back Top Third""","""Torso Back""",1.194905,0.000296,4.798335,0.0,8.886309,0.0,1.743651,1.950777,0.3619048,105,123.6497,1472.01,232.9089,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",99.80404
"""ISIC_0015902""",0,"""IP_4111386""","""65""","""male""","""anterior torso""",3.22,"""TBP tile: close-up""","""3D: XP""",14.242329,12.164757,21.448144,21.121356,25.7462,24.374023,56.414429,60.060388,18.649518,23.314841,6.07994,14.889242,0.51452,2.077572,0.326788,-4.665323,4.783413,6.400196,0.654458,"""Torso Front Top Half""","""Torso Front""",2.481328,21.989453,1.975874,1.771705,9.514499,0.66469,1.258541,1.573733,0.209581,130,-141.02478,1442.185791,58.359802,"""ACEMID MIA""","""CC-0""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",99.989998
"""ISIC_0024200""",0,"""IP_8313778""","""55""","""male""","""anterior torso""",2.73,"""TBP tile: close-up""","""3D: white""",24.72552,20.05747,26.4649,25.71046,36.21798,32.60874,46.94607,52.04118,46.27631,54.85574,2.101708,19.90256,0.0,4.668053,0.7544338,-8.579431,9.148495,6.531302,0.9464478,"""Torso Front Top Half""","""Torso Front""",0.929916,0.001379,3.658854,0.0,6.467562,0.0,2.085409,2.480509,0.3134328,20,-72.31564,1488.72,21.42896,"""Memorial Sloan Kettering Cance…","""CC-BY""","""""","""Benign""","""Benign""","""""","""""","""""","""""","""""","""NA""",70.44251


In [8]:
tam_polars = train_metadados.estimated_size() / (1024 ** 2)
print(f"RAM consumida pelo DataFrame no Polars: {tam_polars:.2f} MB")

RAM consumida pelo DataFrame no Polars: 169.83 MB


In [9]:
tam_pandas = train_metadados_pd.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"RAM consumida pelo DataFrame no Pandas: {tam_pandas:.2f} MB")

RAM consumida pelo DataFrame no Pandas: 511.97 MB


Uma diferença um tanto quanto ignorante.

In [10]:
# limpando a memoria
del tam_pandas, train_metadados_pd

# 2.EDA

In [11]:
# vou testar o metodo lazy do polars nesse projeto
train_metadados = pl.scan_csv(
    RAW_DATA_DIR / 'train-metadata.csv',
    null_values = ["", "NA", "NAN", "N/A", "null", "None"])

## 2.1.Visão Geral

In [12]:
train_metadados.head().collect()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,i64,str,i64,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""ISIC_0015670""",0,"""IP_1235828""",60,"""male""","""lower extremity""",3.04,"""TBP tile: close-up""","""3D: white""",20.244422,16.261975,26.922447,23.954773,33.684638,28.953117,53.058545,55.828924,54.367448,62.025701,3.152561,27.47617,0.0,3.982447,2.967674,-7.658253,8.360566,5.784302,0.901302,"""Right Leg - Upper""","""Right Leg""",1.543016,0.002629,7.09136,0.0,9.307003,0.0,2.036195,2.63778,0.590476,85,-182.703552,613.493652,-42.427948,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,97.517282
"""ISIC_0015845""",0,"""IP_8170065""",60,"""male""","""head/neck""",1.1,"""TBP tile: close-up""","""3D: white""",31.71257,25.36474,26.331,24.54929,41.21903,35.29926,39.70291,44.06404,48.86152,55.36236,0.9194971,12.23529,0.0,6.34783,1.781713,-6.500838,6.839008,4.987244,0.639885,"""Head & Neck""","""Head & Neck""",0.8219178,1.3343e-7,2.116402,0.0,3.354148,0.0,0.8532267,3.912844,0.2857143,55,-0.078308,1575.687,57.1745,"""Memorial Sloan Kettering Cance…","""CC-BY""","""IL_6727506""","""Benign""","""Benign""",null,null,null,null,null,null,3.141455
"""ISIC_0015864""",0,"""IP_6724798""",60,"""male""","""posterior torso""",3.4,"""TBP tile: close-up""","""3D: XP""",22.57583,17.12817,37.97046,33.48541,44.17492,37.6118,59.26585,62.90973,53.96118,61.67052,3.265153,24.18462,0.0,5.447655,4.485044,-7.709336,9.092376,6.290359,0.932147,"""Torso Back Top Third""","""Torso Back""",1.194905,0.000296,4.798335,0.0,8.886309,0.0,1.743651,1.950777,0.3619048,105,123.6497,1472.01,232.9089,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,99.80404
"""ISIC_0015902""",0,"""IP_4111386""",65,"""male""","""anterior torso""",3.22,"""TBP tile: close-up""","""3D: XP""",14.242329,12.164757,21.448144,21.121356,25.7462,24.374023,56.414429,60.060388,18.649518,23.314841,6.07994,14.889242,0.51452,2.077572,0.326788,-4.665323,4.783413,6.400196,0.654458,"""Torso Front Top Half""","""Torso Front""",2.481328,21.989453,1.975874,1.771705,9.514499,0.66469,1.258541,1.573733,0.209581,130,-141.02478,1442.185791,58.359802,"""ACEMID MIA""","""CC-0""",null,"""Benign""","""Benign""",null,null,null,null,null,null,99.989998
"""ISIC_0024200""",0,"""IP_8313778""",55,"""male""","""anterior torso""",2.73,"""TBP tile: close-up""","""3D: white""",24.72552,20.05747,26.4649,25.71046,36.21798,32.60874,46.94607,52.04118,46.27631,54.85574,2.101708,19.90256,0.0,4.668053,0.7544338,-8.579431,9.148495,6.531302,0.9464478,"""Torso Front Top Half""","""Torso Front""",0.929916,0.001379,3.658854,0.0,6.467562,0.0,2.085409,2.480509,0.3134328,20,-72.31564,1488.72,21.42896,"""Memorial Sloan Kettering Cance…","""CC-BY""",null,"""Benign""","""Benign""",null,null,null,null,null,null,70.44251


In [13]:
# qtd linhas do df
qtd_linhas = train_metadados.select(pl.len()).collect().item()
print(f'Número de linhas do dataframe: {qtd_linhas} linhas.')

Número de linhas do dataframe: 401059 linhas.


In [14]:
# contagem de tipos
print('Tipos dos dados: ')
Counter(list(train_metadados.collect_schema().values()))

Tipos dos dados: 


Counter({Float64: 33, String: 19, Int64: 3})

In [34]:
print('Distribuicao de nulos do Dataset:')
nulos = fe.pl_dist_nulos(train_metadados)
nulos

Distribuicao de nulos do Dataset:


variaveis,qtd_nulos,percent_nulos
str,u32,f64
"""iddx_5""",401058,100.0
"""mel_mitotic_index""",401006,99.99
"""mel_thick_mm""",400996,99.98
"""iddx_4""",400508,99.86
"""iddx_3""",399994,99.73
"""iddx_2""",399991,99.73
"""lesion_id""",379001,94.5
"""sex""",11517,2.87
"""anatom_site_general""",5756,1.44


Não irei me aprofundar na análise de nulos para esse projeto, portanto, apenas irei optar por excluir as colunas com mais de 40% dos valores nulos.

In [17]:
# extraindo colunas com muitos nulos
cols_null_drop = (
    nulos
        .filter(pl.col('percent_nulos') >= 40)
        .get_column('variaveis')
        .to_list()
)

print(cols_null_drop)

['iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'iddx_4', 'iddx_3', 'iddx_2', 'lesion_id']


In [18]:
# estatistica do df
train_metadados.describe()

statistic,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
str,str,f64,str,f64,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,f64
"""count""","""401059""",401059.0,"""401059""",398261.0,"""389542""","""395303""",401059.0,"""401059""","""401059""",401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,"""401059""","""401059""",401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,401059.0,"""401059""","""401059""","""22058""","""401059""","""401059""","""1068""","""1065""","""551""","""1""","""53""","""63""",401059.0
"""null_count""","""0""",0.0,"""0""",2798.0,"""11517""","""5756""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""379001""","""0""","""0""","""399991""","""399994""","""400508""","""401058""","""401006""","""400996""",0.0
"""mean""",null,0.00098,null,58.012986,null,null,3.930827,null,null,19.974007,14.919247,28.281706,26.913015,34.786341,30.921279,54.653689,60.996869,42.289976,51.199529,8.539975,19.084517,1.070408,5.05476,1.368691,-8.909552,9.455766,7.538922,0.741238,null,null,2.539773,38.520265,3.451523,3.09159,11.878891,1.016459,2.71519,2.238605,0.306823,86.332073,-3.091862,1039.598221,55.823389,null,null,null,null,null,null,null,null,null,null,null,97.162204
"""std""",null,0.031288,null,13.596165,null,null,1.743068,null,null,3.999489,3.529384,5.278676,4.482994,5.708469,4.829345,5.520849,5.631909,10.864572,12.100527,9.679312,5.357863,0.762918,2.642928,2.215871,3.463362,3.466703,2.404951,0.143857,null,null,1.173169,41.480936,1.724557,2.044813,5.919302,0.734631,1.738165,0.623884,0.125038,52.559511,197.257995,409.819653,87.968245,null,null,null,null,null,null,null,null,null,null,null,8.995782
"""min""","""ISIC_0015670""",0.0,"""IP_0008821""",5.0,"""female""","""anterior torso""",1.0,"""TBP tile: close-up""","""3D: XP""",-2.487115,-9.080269,-0.730989,9.237066,3.054228,11.84652,-1.574164,28.43649,4.223007,17.70226,0.431601,10.761634,0.0,-13.81985,-18.088106,-38.75614,2.952675,3.001138,0.027667,"""Head & Neck""","""Head & Neck""",0.273973,0.0,0.5894259,0.0,2.579237,0.0,0.26816,0.636247,0.052034,0.0,-624.870728,-1052.134,-291.890442,"""ACEMID MIA""","""CC-0""","""IL_0001200""","""Benign""","""Benign""","""Benign adnexal epithelial prol…","""Angiofibroma""","""Actinic keratosis, Bowenoid""","""Blue nevus, Cellular""","""0/mm^2""","""0.2""",1.2611e-16
"""25%""",null,0.0,null,50.0,null,null,2.84,null,null,17.330821,12.46974,24.70438,23.84813,31.003187,27.65829,51.566295,57.29765,34.30339,42.27451,4.109589,15.426205,0.574909,3.413109,0.1110402,-10.67545,6.952381,5.748166,0.6566313,null,null,1.812707,0.10982,2.14384,1.728782,8.338364,0.5638921,1.456571,1.834745,0.2114286,40.0,-147.021851,746.521545,-8.96228,null,null,null,null,null,null,null,null,null,null,null,99.66883
"""50%""",null,0.0,null,60.0,null,null,3.37,null,null,19.80191,14.71393,28.17157,26.701704,3

## 2.2.Análises Univariadas

In [19]:
# checando dist da coluna target
fe.pl_contagem_valores(train_metadados, 'target', excluir_na=False).collect()

target,qtd_target,percent_target
i64,u32,f64
1,393,0.1
0,400666,99.9


Temos um problema extremamente desbalanceado.

In [20]:
# checando distribuicao da coluna sex
fe.pl_contagem_valores(train_metadados, 'sex', excluir_na=False).collect()


sex,qtd_sex,percent_sex
str,u32,f64
null,11517,2.87
"""female""",123996,30.92
"""male""",265546,66.21


In [21]:
# describe da idade
df_idade_train = train_metadados.select(pl.col('age_approx'))
df_idade_train.describe()

statistic,age_approx
str,f64
"""count""",398261.0
"""null_count""",2798.0
"""mean""",58.012986
"""std""",13.596165
"""min""",5.0
"""25%""",50.0
"""50%""",60.0
"""75%""",70.0
"""max""",85.0


Temos um coeficiente de variação (std / mean) de aproximadamente 23.5%, o que indica uma alta variabilidade nos dados. Irei plotar um histograma para compreender melhor essa variável, mas espero uma curva a esquerda.

In [22]:
# plotagem
(
    alt.Chart(df_idade_train.collect())
    .mark_bar()
    .encode(
        x=alt.X("age_approx", title="Idade").bin(maxbins=20),
        y=alt.Y("count()", title="Quantidade"),
    )
    .properties(width=600, height=400, title="Distribuição de Idade")
)

alt.Chart(...)

Nossa base de dados está concentrada principalmente no público entre 50 e 70 anos, o que pode enviesar nosso modelo. Ao cruzar essa variável com a target, precisarei verificar seu poder preditivo afim de pensar em como tratar esse problema.

In [23]:
train_metadados.select(pl.all().n_unique()).collect()

isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_location,tbp_lv_location_simple,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
401059,2,1042,17,3,6,1758,1,2,386052,385304,389890,387763,390703,388865,389798,390743,395726,396358,8029,167648,371189,398257,398886,396505,395511,396877,396648,21,8,75669,395242,391372,368742,9492,367566,396459,390080,74825,36,398446,382410,392817,7,3,22059,52,3,15,26,27,2,8,20,131480


In [24]:
valores_unicos = (
    train_metadados
        .select(pl.all().n_unique())
        .unpivot(variable_name='variaveis', value_name='qtd_valores_unicos')
        .sort('qtd_valores_unicos', descending=False)
    ).collect()
valores_unicos

variaveis,qtd_valores_unicos
str,u32
"""image_type""",1
"""target""",2
"""tbp_tile_type""",2
"""iddx_5""",2
"""sex""",3
…,…
"""tbp_lv_deltaLBnorm""",396877
"""tbp_lv_deltaA""",398257
"""tbp_lv_x""",398446


image_type também pode ser dropada.

In [25]:
train_metadados.select(pl.col('image_type')).unique().collect()

image_type
str
"""TBP tile: close-up"""


In [26]:
train_metadados.select(pl.col('tbp_tile_type')).unique().collect()

tbp_tile_type
str
"""3D: white"""
"""3D: XP"""


In [27]:
train_metadados.select(pl.col('copyright_license')).unique().collect()

copyright_license
str
"""CC-BY-NC"""
"""CC-BY"""
"""CC-0"""


In [28]:
train_metadados.select(pl.col('attribution')).unique().collect()

attribution
str
"""University Hospital of Basel"""
"""ACEMID MIA"""
"""ViDIR Group, Department of Der…"
"""Memorial Sloan Kettering Cance…"
"""Department of Dermatology, Uni…"
"""Frazer Institute, The Universi…"
"""Department of Dermatology, Hos…"


As variáveis: tbp_lv_x, tbp_lv_y, tbp_lv_z fornecem a posição exata da lesão e isso depende muito da estatura do paciente. 

Para inserir a localização na aprendizagem do modelo, irei trabalhar com outras variáveis, as categóricas: anatom_site_general, tbp_lv_location ou tbp_lv_location_simple.

## 2.3.Data Cleaning

In [51]:
## DROPANDO COLUNAS NULAS E NAO PREDITIVAS
# selecionando as colunas a serem dropadas
columns_to_drop = [
    'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',  # vars de posicao  (nao preditivas tambem)
    'attribution', 'copyright_license', 'image_type',  # nao preditivas
    # 'anatom_site_general', 'sex'
      
] + cols_null_drop # nulas

# dropando
df_clean = train_metadados.drop(columns_to_drop)
print(f'Colunas dropadas: {columns_to_drop}')

Colunas dropadas: ['tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'attribution', 'copyright_license', 'image_type', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'iddx_4', 'iddx_3', 'iddx_2', 'lesion_id']


In [ ]:
# calculando quantos registros de target 1 
# perderemos em caso de exclusao de alguns nulos

# calculando qtd de target 1 no dataset de treino
col_nul = ['sex', 'anatom_site_general', 'age_approx']
qtd_target_1_geral = fe.pl_contagem_valores(train_metadados, 'target').filter(pl.col('target') == 1).select('qtd_target').collect().item()

# calculo de perda
for c in col_nul:
    aux = df_clean.filter(~pl.col(c).is_null())
    qtd_target_1_filtrado = (
        fe.pl_contagem_valores(aux, 'target')  # calculando qtd de target 1
            .filter(pl.col('target') == 1)
            .select('qtd_target')
            .collect()
            .item()
        )
    perdidos = qtd_target_1_geral - qtd_target_1_filtrado
    prop_perdidos = perdidos / qtd_target_1_geral
    print(f'Excluindo nulos de {c} perderemos {perdidos} ({prop_perdidos:.2%}) registros de target = 1.')

Excluindo nulos de sex perderemos 10 (2.54%) registros de target = 1.
Excluindo nulos de anatom_site_general perderemos 0 (0.00%) registros de target = 1.
Excluindo nulos de age_approx perderemos 3 (0.76%) registros de target = 1.


Por se tratar de um problema desbalanceado, irei preservar ao máximo a quantidade de registros de target = 1. Portanto, irei preencher sex com a moda e idade com a mediana.

Irei dropar os registros sem anatom_site_general pois pode ser perigoso inferir o local da lesão tanto para o aprendizado do modelo quanto para o paciente.

# Lembrar de preencher os nulos e salvar os valores que serão usados para preencher esses nulos em um .json para poder inserir posteriormente na pipeline final.

## 2.3.Analise Bivariada

In [31]:
# amostrando para analises
mdt_sample = train_metadados.collect().sample(fraction=0.25)
qtd_linhas = mdt_sample.select(pl.col('target')).count().item()
print(f'Dataframe de treino amostrado em 25% resultou em: {qtd_linhas} linhas.')

Dataframe de treino amostrado em 25% resultou em: 100264 linhas.


In [32]:
for t in [0, 1]:
    df_aux = mdt_sample.filter(pl.col('target') == t)
    graf = (
        alt.Chart(df_aux)
        .mark_bar()
        .encode(
            x=alt.X("age_approx", title="Idade").bin(maxbins=20),
            y=alt.Y("count()", title="Quantidade"),
        )
        .properties(width=400, height=200, title=f"Distribuição de Idade com Target = {t}")
    )
    graf.show()


alt.Chart(...)

alt.Chart(...)

In [33]:
float_vars = (
    mdt_sample
    .select(vars_preditivas, cs.float())
    .collect_schema()
    .names()
)[1:]

NameError: name 'vars_preditivas' is not defined

In [ ]:
# amostrando dados para correlacao
#mdt_sample = df_mdt.select(float_vars).collect().sample(fraction=0.15)
#print(f'A amostragem foi feita usando 15% dos dados, o que resultou em: {mdt_sample["target"].count()} linhas.')

# calculando correlacao
df_corr = mdt_sample.select(float_vars).corr()
# del df_mdt_sample

In [ ]:
df_corr

clin_size_long_diam_mm,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_x,tbp_lv_y,tbp_lv_z,tbp_lv_dnn_lesion_confidence
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,-0.040503,-0.074403,-0.116835,-0.036279,-0.106181,-0.055269,-0.071412,0.047859,-0.07194,-0.003041,0.901321,0.480571,0.497963,0.038045,-0.205269,-0.215397,0.202113,0.236492,0.123966,0.85895,0.062274,0.274101,0.500476,0.964939,0.465779,0.236464,0.176359,0.024836,0.00441,0.022439,-0.009722,-0.075041
-0.040503,1.0,0.759606,0.413784,0.48838,0.734384,0.673823,-0.574028,-0.420637,0.304837,0.362087,-0.020229,-0.077862,0.095767,0.497588,-0.001857,-0.308846,0.304087,0.116215,-0.053623,-0.018032,0.055258,-0.082037,0.09417,-0.044318,0.08353,0.192901,0.141233,-0.071443,0.011007,0.218791,0.028799,0.147533
-0.074403,0.759606,1.0,0.243284,0.355481,0.494621,0.650902,-0.510664,-0.750225,0.146347,0.096981,-0.094955,0.164175,-0.167879,-0.18618,-0.139489,0.120563,-0.128426,-0.249178,0.098577,-0.109103,-0.261587,0.203553,-0.155531,-0.053162,-0.129379,-0.184696,0.161297,0.210842,0.018571,0.204062,0.05154,-0.02525
-0.116835,0.413784,0.243284,1.0,0.90997,0.920062,0.826763,0.490644,0.383555,0.51919,0.508734,-0.1019,-0.128714,-0.037789,0.300625,0.542928,-0.148348,0.179949,-0.064641,0.016665,-0.122862,0.173372,-0.103743,-0.039893,-0.137362,-0.041455,0.051881,-0.021202,-0.06077,0.001401,0.31395,0.097588,0.230464
-0.036279,0.48838,0.355481,0.90997,1.0,0.885599,0.939698,0.335323,0.332753,0.427861,0.450479,-0.027068,-0.073013,0.070635,0.263678,0.145814,-0.231515,0.243582,0.029015,0.018862,-0.047475,0.132925,-0.064732,0.060314,-0.054886,0.050204,0.149066,0.047959,-0.044826,0.001933,0.315817,0.117655,0.158634
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.024836,-0.071443,0.210842,-0.06077,-0.044826,-0.077489,0.038309,0.016894,-0.238062,-0.050917,-0.183337,-0.123053,0.711905,-0.316862,-0.389127,-0.054204,0.481364,-0.482931,-0.480968,0.359891,-0.129157,-0.564778,0.92483,-0.332447,0.066472,-0.295755,-0.42353,0.182982,1.0,0.004391,-0.004789,-0.001596,-0.279316
0.00441,0.011007,0.018571,0.001401,0.001933,0.005928,0.008632,-0.010152,-0.017583,0.002649,0.001306,0.001137,0.009688,-0.002469,-0.00814,-0.000573,0.003756,-0.003069,-0.005642,0.006093,0.001847,-0.014283,0.007688,-0.001611,0.005,-0.000471,-0.004344,0.009184,0.004391,1.0,0.004614,-0.00999,-0.0082
0.022439,0.218791,0.204062,0.31395,0.315817,0.327899,0.331675,0.078788,0.027314,0.050213,0.055714,0.011415,0.024848,0.020968,0.058365,0.109431,-0.037128,0.046015,0.018973,0.032673,0.004867,0.01929,0.011186,0.020227,0.016396,0.021685,0.023142,-0.004276,-0.004789,0.004614,1.0,-0.012851,0.069038


In [ ]:
df_corr

target,age_approx,clin_size_long_diam_mm,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,tbp_lv_L,tbp_lv_Lext,tbp_lv_areaMM2,tbp_lv_area_perim_ratio,tbp_lv_color_std_mean,tbp_lv_deltaA,tbp_lv_deltaB,tbp_lv_deltaL,tbp_lv_deltaLB,tbp_lv_deltaLBnorm,tbp_lv_eccentricity,tbp_lv_minorAxisMM,tbp_lv_nevi_confidence,tbp_lv_norm_border,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,tbp_lv_dnn_lesion_confidence
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.0,NaN,0.026383,0.015324,0.019001,-0.026975,-0.015003,-0.012204,-0.003525,-0.041243,-0.030675,-0.002477,0.000649,0.041622,0.006706,0.022242,-0.002097,-0.033938,-0.009983,0.010224,0.010117,-0.004397,0.030617,-0.013865,0.005111,0.018626,0.030245,0.021771,0.012016,0.014985,0.002197,0.006737,0.005463,0.010587,-0.009622,-0.056261
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0.026383,NaN,1.0,-0.041073,-0.07052,-0.121732,-0.039632,-0.110042,-0.056545,-0.07493,0.042272,-0.081744,-0.009952,0.90281,0.474814,0.499777,0.031517,-0.209681,-0.220605,0.207673,0.246868,0.119818,0.857971,0.059988,0.262271,0.500208,0.964853,0.464428,0.238647,0.18844,0.010653,0.005423,0.004337,0.022501,-0.006252,-0.078121
0.015324,NaN,-0.041073,1.0,0.756395,0.413652,0.488546,0.734655,0.672388,-0.57296,-0.416851,0.310108,0.367296,-0.022724,-0.075683,0.09271,0.50237,-0.000065,-0.306675,0.301742,0.111952,-0.049914,-0.01898,0.049363,-0.080889,0.091111,-0.044899,0.080127,0.190281,0.139995,-0.071245,0.001362,0.014259,0.217359,0.033455,0.145998
0.019001,NaN,-0.07052,0.756395,1.0,0.244789,0.35667,0.494329,0.651431,-0.507153,-0.749488,0.153217,0.10254,-0.094361,0.173034,-0.165956,-0.185592,-0.136078,0.12239,-0.129402,-0.251424,0.100475,-0.107379,-0.263242,0.209943,-0.154963,-0.048757,-0.12931,-0.185875,0.163599,0.213812,0.010124,0.018014,0.201528,0.05275,-0.028333
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.006737,NaN,0.005423,0.001362,0.010124,0.002249,-0.000551,0.001934,0.003238,0.001766,-0.010055,0.003743,-0.001139,-0.002938,0.009702,-0.01467,-0.011336,0.006458,0.015634,-0.015689,-0.017858,0.029253,-0.00467,-0.018061,0.015045,-0.014774,0.001615,-0.013986,-0.017653,0.00113,0.016428,1.0,0.00399,0.000842,-0.004241,-0.007936
0.005463,NaN,0.004337,0.014259,0.018014,0.00376,0.001938,0.009273,0.008488,-0.012782,-0.017827,0.009593,0.008761,-0.001284,0.014825,0.00303,-0.002391,0.005039,-0.000463,0.002215,-0.003827,0.010695,-0.000852,-0.012877,0.014389,0.001526,0.004217,0.00143,-0.001132,0.015201,0.01157,0.00399,1.0,0.004775,-0.006198,-0.005125
0.010587,NaN,0.022501,0.217359,0.201528,0.313356,0.315501,0.326876,0.330563,0.080278,0.029966,0.051702,0.057712,0.012364,0.022876,0.021987,0.060128,0.109874,-0.038898,0.047376,0.019613,0.030853,0.005769,0.019144,0.008817,0.021495,0.016553,0.021325,0.022376,0.003727,-0.007159,0.000842,0.004775,1.0,-0.00843,0.073889


# lembrar de cruzar idade com a target!!!

In [ ]:
(13.596165 / 58.012986)*100

23.43641646027322

In [ ]:
13.596165 / 58.012986 * 100

23.43641646027322